# Pre-Impact Fall Detection — ম্যানুয়াল ট্রেনিং নোটবুক (Kaggle)

এই নোটবুক **আপনার নিজের হাতে Kaggle-এ মডেল ট্রেন** করার জন্য। ধাপ:

1. **কোড আপলোড**: বাম panel-এর "Upload" বাটনে `fall-sim-code.zip` ফাইলটা ড্র্যাগ-ড্রপ করুন
   (নোটবুক workspace-এ এক্সট্রাক্ট হয়ে যাবে)।
2. **ডেটাসেট আপলোড (ঐচ্ছিক)**: `/kaggle/input`-এ নিজের Kaggle Dataset হিসেবে
   `SisFall`, `KFall`, `FallAllD`, `UMAFall` (নাম এগুলোই হতে হবে) আপলোড করুন।
   কোনো ডেটা না দিলে নোটবুক **বিল্ট-ইন ডেমো ডেটায়** ট্রেন করবে (পাইপলাইন প্রমাণের জন্য)।
3. **Run All** → শেষে `fall_deploy_artifacts.zip` ডাউনলোড করুন (left panel → Output)।
4. লোকালে সেই zip-এর ফাইল `fall-sim/models/` ও `fall-sim/output/`-এ রাখুন, তারপর:
   `python run_all.py --stage e4 --pretrained` → ফার্মওয়্যার ফাইল তৈরি + Table IV আপডেট।

বিস্তারিত: `README_KAGGLE.md` (প্রজেক্ট ফোল্ডারে)।

In [ ]:
import os, glob, zipfile

# কোড এক্সট্রাক্ট (আপলোড করা zip থাকলে)
if os.path.isdir("/kaggle/working/fall-sim"):
    os.chdir("/kaggle/working/fall-sim")
else:
    for z in glob.glob("/kaggle/working/*.zip"):
        if "fall-sim" in z and "artifacts" not in z:
            zipfile.ZipFile(z).extractall("/kaggle/working")
    os.chdir("/kaggle/working/fall-sim")
print("cwd:", os.getcwd())

In [ ]:
# FALL_RAW-এ আপলোড করা ডেটাসেট ম্যাপ করা (ক্যানোনিক্যাল নামে)
# /kaggle/input read-only, তাই symlink তৈরি হবে /kaggle/working/fall_input-এ
os.environ["FALL_RAW"] = "/kaggle/working/fall_input"
os.makedirs("/kaggle/working/fall_input", exist_ok=True)
canon = {"sisfall": "SisFall", "kfall": "KFall", "fallalld": "FallAllD", "umafall": "UMAFall"}
for d in sorted(glob.glob("/kaggle/input/*")):
    slug = os.path.basename(d).lower()
    for key, name in canon.items():
        if key in slug and not os.path.exists("/kaggle/working/fall_input/" + name):
            try:
                os.symlink(d, "/kaggle/working/fall_input/" + name)
                print("mapped:", os.path.basename(d), "->", name)
            except OSError as e:
                print("skip:", os.path.basename(d), e)
print("available inputs:", sorted(os.listdir("/kaggle/working/fall_input")) or "none (demo data will be used)")

In [ ]:
# ট্রেনিং + কোয়ান্টাইজেশন + আর্টিফ্যাক্ট (ডেটাসেট না থাকলে ডেমো ডেটায় চলবে)
# --epochs 40  = Kaggle-এ দ্রুত; পেপার-কোয়ালিটির জন্য 100 দিন
!cd /kaggle/working/fall-sim && python tools/kaggle_train.py --datasets SisFall KFall FallAllD --epochs 40 --patience 8

In [ ]:
import os
print(open("/kaggle/working/fall-sim/output/kaggle_summary.json").read())
msg = "\nডাউনলোড: বাম panel-এর Output ফোল্ডার → fall_deploy_artifacts.zip"
msg += "\n(পাথ: /kaggle/working/fall-sim/fall_deploy_artifacts.zip)\n"
print(msg)

## পরের ধাপ (লোকাল মেশিনে)

```bash
# 1. artifacts আনজিপ করে এই জায়গায় রাখুন
fall-sim/models/proposed_fp32.keras
fall-sim/models/model.tflite
fall-sim/output/frozen_normalization.json
fall-sim/output/kaggle_summary.json

# 2. নতুন ট্রেন না করেই ফার্মওয়্যার ফাইল রি-এক্সপোর্ট + Table IV আপডেট
cd fall-sim
python run_all.py --stage e4 --pretrained

# 3. ESP32-তে ফ্ল্যাশ
#    firmware/fall_detector/ → Arduino IDE → Upload
```

> সতর্কতা: `frozen_normalization.json` এবং `model.tflite` **একই ট্রেনিং রান** থেকে আসে —
> জোড়া ভাঙবেন না (প্ল্যান §4.1-এর গোল্ডেন রুল: ফার্মওয়্যার আর ট্রেনিং-এর preprocess
> হুবহু এক হতে হবে)।